# Can we fit $\mathcal{H}$ Using DNNs at a Single Kinematic Point Using the KM15/BKM10 Formalism for the Cross-Section?

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Libraries:

In [ ]:
# native libraries
import glob
import gc
import datetime
import yaml
import re
import time
from pathlib import Path

# data analysis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import corner
from scipy.stats import norm
from sklearn.model_selection import train_test_split

# formalism
import gepard as g
from gepard.fits import th_KM15

# BKM10 library stuff:
from bkm10_lib.core import DifferentialCrossSection
from bkm10_lib.inputs import BKM10Inputs
from bkm10_lib.cff_inputs import CFFInputs

# local imports
from unfolded_loss import UnfoldedSimultaneousFitLoss
from simultaneous_fit_dnn_config import bkm10_cross_section
from simultaneous_fit_dnn_config import bkm10_bsa

### (1.2): Matplotlib Plotting Aesthetics:

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "savefig.dpi": 300,
    "axes.labelsize": 16,
    "xtick.direction": "in",
    "xtick.major.size": 8.5,
    "xtick.major.width": 1.0,
    "xtick.minor.size": 4.5,
    "xtick.minor.width": 1.0,
    "xtick.minor.visible": True,
    "xtick.top": True,
    "ytick.direction": "in",
    "ytick.major.size": 8.5,
    "ytick.major.width": 1.0,
    "ytick.minor.size": 4.5,
    "ytick.minor.width": 1.0,
    "ytick.minor.visible": True,
    "ytick.right": True,
    "xtick.labelsize": 15.0,
    "ytick.labelsize": 15.0,
})

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")
print(f"[INFO]: corner version: {corner.__version__}")

### (1.4): TensorFlow Configurations:

In [ ]:
cpus = tf.config.list_physical_devices('CPU')
gpus = tf.config.list_physical_devices('GPU')

print(f"[INFO]: Number of CPUs Available: {len(cpus)}")
print(f"[INFO]: Number of GPUs Available: {len(gpus)}")
print(f"[INFO]: The available devices are: {tf.config.list_physical_devices()}")
print(f"[INFO]: Is CUDA-built? {tf.test.is_built_with_cuda()}")

if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    
# [NOTE]: major optimization issue in running this line:
# tf.config.run_functions_eagerly(True)

## (2): Generating Pseudodata:

### (2.0): Data Collection Definitions:

In [ ]:
with open("closure_test_config.yml", "r") as file:
    config = yaml.safe_load(file)

VERSION_NUMBER = config["versioning"]["major"]
MINOR_NUMBER = config["versioning"]["minor"]
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

NUMBER_OF_EPOCHS = config["dnn_config"]["epochs"]
NUMBER_OF_REPLICAS = config["dnn_config"]["replicas"]
BATCH_SIZE = config["dnn_config"]["batch_size"]
LEARNING_RATE = config["dnn_config"]["adam_learning_rate"]

# CFF settings
IS_CFF_REAL_H_FREE = config["cff_config"]["enable_cff_real_h"]
IS_CFF_IMAG_H_FREE = config["cff_config"]["enable_cff_imag_h"]
IS_CFF_REAL_HT_FREE = config["cff_config"]["enable_cff_real_ht"]
IS_CFF_IMAG_HT_FREE = config["cff_config"]["enable_cff_imag_ht"]
IS_CFF_REAL_E_FREE = config["cff_config"]["enable_cff_real_e"]
IS_CFF_IMAG_E_FREE = config["cff_config"]["enable_cff_imag_e"]
IS_CFF_REAL_ET_FREE = config["cff_config"]["enable_cff_real_et"]
IS_CFF_IMAG_ET_FREE = config["cff_config"]["enable_cff_imag_et"]

# cross-section observables:
IS_UNP_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_unp_beam_unp_target_xsec"]
IS_PLUS_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_plus_beam_unp_target_xsec"]
IS_MINUS_BEAM_UNP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_minus_beam_unp_target_xsec"]
IS_UNP_BEAM_LP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_unp_beam_lp_target_xsec"]
IS_PLUS_BEAM_LP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_plus_beam_lp_target_xsec"]
IS_MINUS_BEAM_LP_TARGET_XSEC_INCLUDED = config["observable_config"]["enable_minus_beam_lp_target_xsec"]

IS_UNP_TARGET_BSA_INCLUDED = config["observable_config"]["enable_unp_target_bsa"]
IS_PLUS_TARGET_BSA_INCLUDED = config["observable_config"]["enable_plus_target_bsa"]
IS_MINUS_TARGET_BSA_INCLUDED = config["observable_config"]["enable_minus_target_bsa"]

IS_UNP_BEAM_TSA_INCLUDED = config["observable_config"]["enable_unp_beam_tsa"]
IS_PLUS_BEAM_TSA_INCLUDED = config["observable_config"]["enable_plus_beam_tsa"]
IS_MINUS_BEAM_TSA_INCLUDED = config["observable_config"]["enable_minus_beam_tsa"]

IS_DSA_INCLUDED = config["observable_config"]["enable_dsa"]

enabled_observables = []

if IS_UNP_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("unp_beam_unp_target_xsec")

if IS_PLUS_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("plus_beam_unp_target_xsec")

if IS_MINUS_BEAM_UNP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("minus_beam_unp_target_xsec")

if IS_UNP_BEAM_LP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("unp_beam_lp_target_xsec")

if IS_PLUS_BEAM_LP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("plus_beam_lp_target_xsec")

if IS_MINUS_BEAM_LP_TARGET_XSEC_INCLUDED:
    enabled_observables.append("minus_beam_lp_target_xsec")

if IS_UNP_TARGET_BSA_INCLUDED:
    enabled_observables.append("unp_target_bsa")

if IS_PLUS_TARGET_BSA_INCLUDED:
    enabled_observables.append("plus_lp_target_bsa")

if IS_MINUS_TARGET_BSA_INCLUDED:
    enabled_observables.append("minus_lp_target_bsa")

if IS_UNP_BEAM_TSA_INCLUDED:
    enabled_observables.append("unp_beam_tsa")

if IS_PLUS_BEAM_TSA_INCLUDED:
    enabled_observables.append("plus_beam_tsa")

if IS_MINUS_BEAM_TSA_INCLUDED:
    enabled_observables.append("minus_beam_tsa")

if IS_DSA_INCLUDED:
    enabled_observables.append("dsa")

OBSERVABLE_WEIGHTS = [
    0.5,
    0.5,
]

### (2.0): Paths and Directories:

In [ ]:
base_directory = Path('./local') / f"version_{MAJOR_MINOR_NUMBER}"

data_directory = base_directory / "data"
plots_directory = base_directory / "plots"
replica_directory = base_directory / "replica"
learning_curves_directory = base_directory / "learning_curves"

data_directory.mkdir(parents = True, exist_ok = True)
plots_directory.mkdir(parents = True, exist_ok = True)
replica_directory.mkdir(parents = True, exist_ok = True)
learning_curves_directory.mkdir(parents = True, exist_ok = True)

### (2.1): Defining the Azimuthal Range:

In [ ]:
STARTING_PHI_VALUE_IN_DEGREES = config["data_config"]["start_value_of_phi_in_degrees"]
ENDING_PHI_VALUE_IN_DEGREES = config["data_config"]["end_value_of_phi_in_degrees"]
NUMBER_OF_PHI_POINTS = config["data_config"]["number_of_phi_points"] + 1

phi_array_in_degrees = np.linspace(
    start = STARTING_PHI_VALUE_IN_DEGREES,
    stop = ENDING_PHI_VALUE_IN_DEGREES,
    num = NUMBER_OF_PHI_POINTS)

phi_array_in_radians = [np.radians(degree_value) for degree_value in phi_array_in_degrees]

print(f"[INFO]: New list of {len(phi_array_in_radians)} of azimuthal angles from {STARTING_PHI_VALUE_IN_DEGREES} degrees to {ENDING_PHI_VALUE_IN_DEGREES} degrees.")

## (4): DNN Model Definitions:

We need to go through all of this section in order to get TensorFlow working with the BKM10 custom computation for the cross-section and BSA.

### (4.1): Static Quantities Used in Loss Computation:

In [ ]:
_CONVERSION_GEV6_GEV4NB = .389379 * 1000000.
_MASS_OF_PROTON_IN_GEV = 0.93827208816
_QED_FINE_STRUCTURE = 1./137.035999177
_ELECTRIC_FORM_FACTOR_CONSTANT = 0.710649
_PROTON_MAGNETIC_MOMENT = 2.79284734463

def compute_fe(t):
    return 1./ (1. - (t / _ELECTRIC_FORM_FACTOR_CONSTANT))**2

def compute_fg(fe):
    return _PROTON_MAGNETIC_MOMENT * fe

def compute_f2(t, fe, fg):
    tau = -1. * t/ (4. * _MASS_OF_PROTON_IN_GEV**2)
    numerator = fg - fe
    denominator = 1. + tau
    return numerator / denominator

def compute_f1(fg, f2):
    return fg - f2

def compute_epsilon(xb, q_squared):
    return 2. * xb * _MASS_OF_PROTON_IN_GEV / tf.sqrt(q_squared)

def compute_y(k_beam, q_squared, ep):
    return tf.sqrt(q_squared) / (ep * k_beam)

def compute_skewness(xb, t, q_squared):
    return xb * (1. + (t / (2. * q_squared))) / (2. - xb + (xb * t / q_squared))

def compute_t_min(xb, q_squared, ep):
    return -1. * q_squared * ((2. * (1. - xb) * (1. - tf.sqrt(1. + ep**2))) + ep**2) / ((4. * xb * (1. - xb)) + ep**2)

def compute_t_prime(t, tmin):
    return (t-tmin)

def compute_k_tilde(xb, q_squared, t, tmin, ep):
    return tf.sqrt(tmin - t) * tf.sqrt(((1. - xb) * tf.sqrt((1. + ep**2))) + (((tmin - t) * (ep**2 + (4. * (1. - xb) * xb))) / (4. * q_squared)))

def compute_k(q_squared, y_lep, ep, k_tilde):
    return tf.sqrt(((1. - y_lep + (ep**2 * y_lep**2 / 4.)) / q_squared)) * k_tilde

def compute_k_dot_delta(q_squared, xb, t, phi_azi, ep, y_lep, k):
    return (-1.*q_squared / (2.*y_lep*(1.+ep**2))) * (1. + ((2.*k*tf.cos(tf.constant(np.pi) - phi_azi)) - ((t / q_squared)*(1.-(xb * (2. - y_lep)) + (y_lep * ep**2 / 2.))) + (y_lep * ep**2 / 2.)))

def prop_1(q_squared, kdd):
    return (1. + (2. * (kdd / q_squared)))

def prop_2(q_squared, t, kdd):
    return ((-2. * (kdd / q_squared)) + (t / q_squared))

### (2.2): Defining the Kinematic Point:

#### (2.2.1): Required Kinematics

In [ ]:
FIXED_K = 5.750
FIXED_XB = 0.360
FIXED_T = -0.17
FIXED_Q_SQUARED = 2.300

print(f"[INFO]: Received k = {FIXED_K} GeV")
print(f"[INFO]: Received xB = {FIXED_XB}")
print(f"[INFO]: Received t = {FIXED_T} GeV^2")
print(f"[INFO]: Received Q^2 = {FIXED_Q_SQUARED} GeV^2")

#### (2.2.2): **DON'T FORGET**: Lepton Helicity and Target Polarization

In [ ]:
TEST_LEPTON_HELICITY = 0.0
TEST_TARGET_POLARIZATION = 0.0

### (2.3): Use the KM15 Model of $Re[\mathcal{H}]$ and $Im[\mathcal{H}]$ to Generate a True Value:

Uses `gepard`'s `DataPoint()` class to define the kinematic setting.

In [ ]:
try:
    # [NOTE]: We actually don't need to be super accurate here because we ONLY use
    # this class to evaluate the CFFs later!
    test_datapoints = [g.DataPoint(
        xB = FIXED_XB, t = FIXED_T, Q2 = FIXED_Q_SQUARED, phi = fixed_phi,
        process = "ep2epgamma", exptype = 'fixed target',
        in1energy = FIXED_K, in1charge = -1, in1polarization = +1, in1units = 'rad',
        observable = 'XS',
        fname = 'Trento') for fixed_phi in phi_array_in_radians]
except ZeroDivisionError:
    print(
        f"[ERROR]: Kinematic setting k = {FIXED_K}, xb = {FIXED_XB}, "
        f"t = {FIXED_T}, Q^2 = {FIXED_Q_SQUARED} unphysical according to gepard."
        )

#### (2.3.1): Check out the Length of the Datapoint Array:

These arrays should have exactly the same length without absolutely any difference.

In [ ]:
len(test_datapoints) == len(phi_array_in_radians)

### (2.4): Evaluate all the CFFs using KM15 for the Datapoints 

In [ ]:
# here we compute the actual CFFs!
real_h_values = np.array([th_KM15.ReH(datapoint) for datapoint in test_datapoints])
imag_h_values = np.array([th_KM15.ImH(datapoint) for datapoint in test_datapoints])
real_e_values = np.array([th_KM15.ReE(datapoint) for datapoint in test_datapoints])
imag_e_values = np.array([th_KM15.ImE(datapoint) for datapoint in test_datapoints])
real_ht_values = np.array([th_KM15.ReHt(datapoint) for datapoint in test_datapoints])
imag_ht_values = np.array([th_KM15.ImHt(datapoint) for datapoint in test_datapoints])
real_et_values = np.array([th_KM15.ReEt(datapoint) for datapoint in test_datapoints])
imag_et_values = np.array([th_KM15.ImEt(datapoint) for datapoint in test_datapoints])

#### (2.4.1): Check out the Length of the CFF Data Array:

These arrays should ALSO be of exactly the same length --- For every phi, we have a kinematic point, and every kinematic point will have a CFF value.

In [ ]:
print(len(real_h_values) == len(test_datapoints))
print(len(imag_h_values) == len(test_datapoints))

#### (2.4.2): See Some Values in the CFF Arrays:

High degree of redundancy expected; kinematics don't change, ergo CFFs don't change.

In [ ]:
real_h_values[0:5]

### (2.4.3): Define ("globally") the KM15 CFF values:

In [ ]:
# here we actually set the KM15 values to 0:
CFF_REAL_H_KM15 = real_h_values[0] if IS_CFF_REAL_H_FREE else 0.0
print(f"[INFO]: Setting Re[H] = {CFF_REAL_H_KM15}")
CFF_IMAG_H_KM15 = imag_h_values[0] if IS_CFF_IMAG_H_FREE else 0.0
print(f"[INFO]: Setting Im[H] = {CFF_IMAG_H_KM15}")
CFF_REAL_HT_KM15 = real_ht_values[0] if IS_CFF_REAL_HT_FREE else 0.0
print(f"[INFO]: Setting Re[Ht] = {CFF_REAL_HT_KM15}")
CFF_IMAG_HT_KM15 = imag_ht_values[0] if IS_CFF_IMAG_HT_FREE else 0.0
print(f"[INFO]: Setting Im[Ht] = {CFF_IMAG_HT_KM15}")
CFF_REAL_E_KM15 = real_e_values[0] if IS_CFF_REAL_E_FREE else 0.0
print(f"[INFO]: Setting Re[E] = {CFF_REAL_E_KM15}")
CFF_IMAG_E_KM15 = imag_e_values[0] if IS_CFF_IMAG_E_FREE else 0.0
print(f"[INFO]: Setting Im[E] = {CFF_IMAG_E_KM15}")
CFF_REAL_ET_KM15 = real_et_values[0] if IS_CFF_REAL_ET_FREE else 0.0
print(f"[INFO]: Setting Re[Et] = {CFF_REAL_ET_KM15}")
CFF_IMAG_ET_KM15 = imag_et_values[0] if IS_CFF_IMAG_ET_FREE else 0.0
print(f"[INFO]: Setting Im[Et] = {CFF_IMAG_ET_KM15}")

CFF_H_KM15 = complex(CFF_REAL_H_KM15, CFF_IMAG_H_KM15)
CFF_H_TILDE_KM15 = complex(CFF_REAL_HT_KM15, CFF_IMAG_HT_KM15)
CFF_E_KM15 = complex(CFF_REAL_E_KM15, CFF_IMAG_E_KM15)
CFF_E_TILDE_KM15 = complex(CFF_REAL_ET_KM15, CFF_IMAG_ET_KM15)

### (2.5): Preparing for future "plot strings"

#### (2.5.1): The kinematic setting plot string:

In [ ]:
this_kinematic_set_title_string = (
    rf"$k = {FIXED_K:.3f}$ GeV, "
    rf"$x_B = {FIXED_XB:.3f}$, "
    rf"$t = {FIXED_T:.3f}$ GeV$^2$, "
    rf"$Q^2 = {FIXED_Q_SQUARED:.3f}$ GeV$^2$"
)

#### (2.5.2): The KM15 CFF setting plot string:

In [ ]:
km15_cff_string = (
    rf"$\mathcal{{H}} = {CFF_H_KM15:.3f}$, "
    rf"$\mathcal{{E}} = {CFF_E_KM15:.3f}$, "
    rf"$\widetilde{{\mathcal{{H}}}} = {CFF_H_TILDE_KM15:.3f}$, "
    rf"$\widetilde{{\mathcal{{E}}}} = {CFF_E_TILDE_KM15:.3f}$ "
)

### (2.5): Use the [BKM10 Library](https://pypi.org/project/bkm10/) to Predict the Cross-Section

#### (2.5.2): Use the BKM10 Library to Evaluate the Cross-Section as a Function of $\phi$

In [ ]:
km15_cross_section = DifferentialCrossSection(
    configuration = {
        "kinematics": BKM10Inputs(
            lab_kinematics_k = FIXED_K,
            squared_Q_momentum_transfer = FIXED_Q_SQUARED,
            x_Bjorken = FIXED_XB,
            squared_hadronic_momentum_transfer_t = FIXED_T),
        "cff_inputs": CFFInputs(
            compton_form_factor_h = CFF_H_KM15,
            compton_form_factor_h_tilde = CFF_H_TILDE_KM15,
            compton_form_factor_e = CFF_E_KM15,
            compton_form_factor_e_tilde = CFF_E_TILDE_KM15),
        "using_ww": True
    },
    verbose = False, debugging = False)

##### (2.5.2.1): $d^{4}\sigma^{UU}$ using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = 0.0).real

##### (2.5.2.2): $d^{4}\sigma^{+U}$ using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = +1.0,
    target_polarization = 0.0).real

##### (2.5.2.3): $d^{4}\sigma^{-U}$ using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_unp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = -1.0,
    target_polarization = 0.0).real

##### (2.5.2.4): $d^{4}\sigma^{U+}$ using **KM15** CFFs:

In [ ]:
bkm10_unp_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = +0.5).real

##### (2.5.2.5): $d^{4}\sigma^{++}$ using **KM15** CFFs:

In [ ]:
bkm10_plus_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = +1.0,
    target_polarization = +0.5).real

##### (2.5.2.6): $d^{4}\sigma^{-+}$ using **KM15** CFFs:

In [ ]:
bkm10_minus_beam_lp_target_km15 = km15_cross_section.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = -1.0,
    target_polarization = +0.5).real

##### (2.5.2.7): $\text{BSA}\left( \Lambda = 0 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_km15 = km15_cross_section.compute_bsa(
    phi_array_in_radians,
    target_polarization = 0.0).real

##### (2.5.2.8): $\text{BSA}\left( \Lambda = +1/2 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_plus_lp_target_km15 = km15_cross_section.compute_bsa(
    phi_array_in_radians,
    target_polarization = +0.5).real

##### (2.5.2.9): $\text{BSA}\left( \Lambda = -1/2 \right)$ Cross-Section Values using **KM15** CFFs:

In [ ]:
bkm10_bsa_minus_lp_target_km15 = km15_cross_section.compute_bsa(
    phi_array_in_radians,
    target_polarization = -0.5).real

#### (2.5.3): A Function that Plots all the BKM10-Predicted Observables:

In [ ]:
def plot_bkm10_observable(
    x_data,
    y_data,
    y_label,
    title,
    observable_label,
    figure_filename):
    
    figure, axis = plt.subplots(1, figsize = (10, 10))

    axis.scatter(
        x_data,
        y_data,
        s = 4.0,
        color = "blue",
        label = observable_label
    )

    axis.set_xlabel(
        r"$\phi$ [radians]",
        fontsize = 20.)
    
    axis.set_ylabel(
        y_label,
        fontsize = 20.)
    
    axis.set_title(
        title,
        fontsize = 16.)

    axis.legend(
        fontsize = 20.)
    
    axis.grid(
        visible = True,
        alpha = 0.35)

    for extension in ["png", "eps"]:
        figure.savefig(
            plots_directory / f"{figure_filename}.{extension}",
            facecolor = "white"
        )

    plt.close(figure)

In [ ]:
observables_to_plot = {
    "xsec_UU": {
        "data": bkm10_unp_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{UU}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_unp_beam_unp_target_prediction",
    },
    "xsec_+U": {
        "data": bkm10_plus_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{+U}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_plus_beam_unp_target_prediction",
    },
    "xsec_-U": {
        "data": bkm10_minus_beam_unp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{-U}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_minus_beam_unp_target_prediction",
    },
     "xsec_UL": {
        "data": bkm10_unp_beam_lp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{L}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_unp_beam_lp_target_prediction",
    },
    "xsec_+L": {
        "data": bkm10_plus_beam_lp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{L+}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_plus_beam_lp_target_prediction",
    },
    "xsec_-L": {
        "data": bkm10_minus_beam_lp_target_km15,
        "label": r"BKM10 $d^{4}\sigma^{L-}(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
        "filename": "bkm10_xsec_minus_beam_lp_target_prediction",
    },
    "bsa_LU": {
        "data": bkm10_bsa_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = 0)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = 0)$ [unitless]",
        "filename": "bkm10_bsa_lp_target_prediction",
    },
    "bsa_U+": {
        "data": bkm10_bsa_plus_lp_target_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = +1/2)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = +1/2)$ [unitless]",
        "filename": "bkm10_bsa_plus_lp_target_prediction",
    },
    "bsa_U-": {
        "data": bkm10_bsa_minus_lp_target_km15,
        "label": r"BKM10 $\textrm{BSA}(\Lambda = -1/2)(\mathcal{F}_{\textrm{KM15}})$",
        "ylabel": r"$\textrm{BSA}(\Lambda = -1/2)$ [unitless]",
        "filename": "bkm10_bsa_plus_lp_target_prediction",
    },
}

for observable_name, observable in observables_to_plot.items():

    plot_bkm10_observable(
        phi_array_in_radians,
        observable["data"],
        y_label = observable["ylabel"],
        title = (
            rf"{observable_name} vs. $\phi$, "
            f"{this_kinematic_set_title_string}"    
            "\n"
            f"(KM15): {km15_cff_string}"
        ),
        observable_label = observable["label"],
        figure_filename = observable['filename']
    )

### (2.6): Piece Together Fixed Kinematic Settings, Varying Phi Data, and CFF Data into Pandas DataFrame:

In [ ]:
test_dataframe = pd.DataFrame({
    "set": 1,
    # kinematics
    "k": FIXED_K, "q_squared": FIXED_Q_SQUARED, "x_b": FIXED_XB, "t": FIXED_T,
    "phi": phi_array_in_radians,
    # observables:
    "unp_beam_unp_target_xsec": bkm10_unp_beam_unp_target_km15, "unp_target_bsa": bkm10_bsa_km15,
    # KM15 CFFs:
    "Re[H]": real_h_values, "Im[H]": imag_h_values,
    "Re[E]": real_e_values, "Im[E]": imag_e_values,
    "Re[Ht]": real_ht_values, "Im[Ht]": imag_ht_values,
    "Re[Et]": real_et_values, "Im[Et]": imag_et_values
    },
    index = None)

### (2.7): We now augment the `test_dataframe` with all of the constant physics quantities:

In [ ]:
test_dataframe["fe"] = compute_fe(test_dataframe["t"].values)
test_dataframe["fg"] = compute_fg(test_dataframe["fe"].values)
test_dataframe["f2"] = compute_f2(test_dataframe["t"].values, test_dataframe["fe"].values, test_dataframe["fg"].values)
test_dataframe["f1"] = compute_f1(test_dataframe["fg"].values, test_dataframe["f2"].values)

test_dataframe["ep"] = compute_epsilon(test_dataframe["x_b"].values,test_dataframe["q_squared"].values)
test_dataframe["y"] = compute_y(test_dataframe["k"].values, test_dataframe["q_squared"].values, test_dataframe["ep"].values)
test_dataframe["xi"] = compute_skewness(test_dataframe["x_b"].values, test_dataframe["t"].values, test_dataframe["q_squared"].values)
test_dataframe["t_min"] = compute_t_min(test_dataframe["x_b"].values, test_dataframe["q_squared"].values, test_dataframe["ep"].values)
test_dataframe["t_prime"] = compute_t_prime(test_dataframe["t"].values, test_dataframe["t_min"].values)
test_dataframe["k_tilde"] = compute_k_tilde(test_dataframe["x_b"].values, test_dataframe["q_squared"].values, test_dataframe["t"].values, test_dataframe["t_min"].values, test_dataframe["ep"].values)
test_dataframe["kinematic_k"] = compute_k(test_dataframe["q_squared"].values, test_dataframe["y"].values, test_dataframe["ep"].values, test_dataframe["k_tilde"].values)

test_dataframe["k_dot_delta"] = compute_k_dot_delta(
    test_dataframe["q_squared"].values, test_dataframe["x_b"].values, test_dataframe["t"].values, test_dataframe["phi"].values,
    test_dataframe["ep"].values, test_dataframe["y"].values, test_dataframe["kinematic_k"].values)
test_dataframe["prop_1"] = prop_1(test_dataframe["q_squared"].values, test_dataframe["k_dot_delta"].values)
test_dataframe["prop_2"] = prop_2(test_dataframe["q_squared"].values, test_dataframe["t"].values, test_dataframe["k_dot_delta"])

#### (2.6.1): Checking Out the DataFrame with the Pseudodata:

In [ ]:
test_dataframe.head()

## (3): Data Preprocessing:

### (3.1): Define What Becomes the $x$ and $y$ Data:

**Important**: Define here what you want to train on!

[NOTE]: Data is now NumPy arrays.

In [ ]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec", "unp_target_bsa"]]

TOTAL_DATA_SIZE = len(x_data)
print(f"[NOTE]: Total data size is: {TOTAL_DATA_SIZE}")

### (3.2): Train-Test Split Here:

#### (3.2.1): Define the Partition Percentages Here (Train/Validate/Test):

In [ ]:
# https://stackoverflow.com/a/13623707 -> for the typical train/val/test data split
# (we are *not* using the same numbers as recommended here)

_DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE = 0.1 # 90% temporary, 10% testing
_DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE = 0.1 # of the above 90% temporary, 90% training, 10% validation

number_of_dnn_testing_points = np.ceil(TOTAL_DATA_SIZE * _DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE)
number_of_dnn_temporary_points = TOTAL_DATA_SIZE - number_of_dnn_testing_points
number_of_dnn_validation_points = np.ceil(number_of_dnn_temporary_points * _DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE)
number_of_dnn_training_points = number_of_dnn_temporary_points - number_of_dnn_validation_points

print(f"[NOTE]: Testing/Temporary Split is {_DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE * 100}%, giving {number_of_dnn_testing_points} testing points")
print(f"[NOTE]: Training/Validation Split is {_DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE * 100}%, giving {number_of_dnn_validation_points} validation points")
print(f"[NOTE]: Remaining training data points are: {number_of_dnn_training_points}")

In [ ]:
k = np.full(NUMBER_OF_PHI_POINTS, FIXED_K, dtype = np.float32)
t = np.full(NUMBER_OF_PHI_POINTS, FIXED_T, dtype = np.float32)
xb = np.full(NUMBER_OF_PHI_POINTS, FIXED_XB, dtype = np.float32)
q2 = np.full(NUMBER_OF_PHI_POINTS, FIXED_Q_SQUARED, dtype = np.float32)

phi = np.array(phi_array_in_radians, dtype = np.float32)

fe = compute_fe(t)
fg = compute_fg(fe)
f2 = compute_f2(t, fe, fg)
f1 = compute_f1(fg, f2)

epsilon = compute_epsilon(xb,q2)
y = compute_y(k, q2, epsilon)
xi = compute_skewness(xb, t, q2)
t_min = compute_t_min(xb, q2, epsilon)
t_prime = compute_t_prime(t, t_min)
k_tilde = compute_k_tilde(xb, q2, t, t_min, epsilon)
kinematic_k = compute_k(q2, y, epsilon, k_tilde)

k_dot_delta = compute_k_dot_delta(
    q2, xb, t, phi,
    epsilon, y, kinematic_k)
prop_1_values = prop_1(q2, k_dot_delta)
prop_2_values = prop_2(q2, t, k_dot_delta)

helicity = np.full(NUMBER_OF_PHI_POINTS, 0.0, dtype = np.float32)
polarization = np.full(NUMBER_OF_PHI_POINTS, 0.0, dtype = np.float32)

kinematics_and_phi = np.column_stack((t, xb, q2, phi)).astype(np.float32)

physics_data = np.column_stack((
    # kinematics, phi:
    t, xb, q2, phi,

    # form factors:
    fe, fg, f1, f2,

    # derived kinematics:
    epsilon, y, xi, t_prime, k_tilde, kinematic_k,

    # phi-dependent stuff:
    k_dot_delta,  prop_1_values, prop_2_values,

    # polarizations:
    helicity, polarization
)).astype(np.float32)

observable_data = {
    "unp_beam_unp_target_xsec": bkm10_unp_beam_unp_target_km15,
    "unp_target_bsa": bkm10_bsa_km15,
}

# stack the observables to fit:
observables = np.column_stack((
    [observable_data[name] for name in enabled_observables]
)).astype(np.float32)

####################################################################################################
# Constructing the training data
####################################################################################################

indices = np.arange(NUMBER_OF_PHI_POINTS)

dnn_inputs = np.column_stack((t, xb, q2)).astype(np.float32)

training_indices, testing_indices = train_test_split(
    indices,
    test_size = 0.10,
    random_state = 7009,)

training_indices, validation_indices = train_test_split(
    training_indices,
    test_size = 0.10,
    random_state = 7009,)

splits = {
    "train": training_indices,
    "validation": validation_indices,
    "test": testing_indices,
}

x_training = dnn_inputs[training_indices]
x_validation = dnn_inputs[validation_indices]
x_testing = dnn_inputs[testing_indices]

precomputed_physics_training = physics_data[training_indices]
precomputed_physics_validation = physics_data[validation_indices]
precomputed_physics_testing = physics_data[testing_indices]

y_training = observables[training_indices]
y_validation = observables[validation_indices]
y_testing = observables[testing_indices]

### (3.3): What does the pre-processed data look like?

#### (3.3.1): Training $x$ Features:

In [ ]:
print(f"[INFO]: Size of x training: {len(x_training)}")
print(f"[INFO]: Does it match with what's expected? {len(x_training) == number_of_dnn_training_points}")

#### (3.3.2): Training $y$ Labels:

In [ ]:
print(f"[INFO]: Size of y training: {len(y_training)}")
print(f"[INFO]: Does it match with what's expected? {len(y_training) == number_of_dnn_training_points}")

#### (3.3.3): Checking out the Validation $x$ Features:

In [ ]:
print(f"[INFO]: Size of x validation: {len(x_validation)}")
print(f"[INFO]: Does it match with what's expected? {len(x_validation) == number_of_dnn_validation_points}")

#### (3.3.4): Checking out the Validation $y$ Features:

In [ ]:
print(f"[INFO]: Size of y validation: {len(y_validation)}")
print(f"[INFO]: Does it match with what's expected? {len(y_validation) == number_of_dnn_validation_points}")

#### (3.3.5): Checking out the Testing $x$ Features:

In [ ]:
print(f"[INFO]: Size of x testing: {len(x_testing)}")
print(f"[INFO]: Does it match with what's expected? {len(x_testing) == number_of_dnn_testing_points}")

#### (3.3.6): Checking out the Testing $y$ Features:

In [ ]:
print(f"[INFO]: Size of y testing: {len(y_testing)}")
print(f"[INFO]: Does it match with what's expected? {len(y_testing) == number_of_dnn_testing_points}")

### (3.4): Visualizing the Training/Validation/Testing Datapoints:

#### (3.4.1): Extracting the ($\phi$, $d^{4}\sigma$) Data Pairs in Train/Val/Test

In [ ]:
# a numpy array of all the corresponding phi-points
# if you do NOT UNDERSTAND why the code below selects the phi-values that are used in training, please run each piece interactively
x_training_phi_points = x_training
x_validation_phi_points = x_validation
x_testing_phi_points = x_testing

xsecs = DifferentialCrossSection(
    configuration = {
        "kinematics": BKM10Inputs(
            lab_kinematics_k = FIXED_K,
            squared_Q_momentum_transfer = FIXED_Q_SQUARED,
            x_Bjorken = FIXED_XB, 
            squared_hadronic_momentum_transfer_t = FIXED_T),
        "cff_inputs": CFFInputs(
            compton_form_factor_h = CFF_H_KM15,
            compton_form_factor_h_tilde = CFF_H_TILDE_KM15,
            compton_form_factor_e = CFF_E_KM15,
            compton_form_factor_e_tilde = CFF_E_TILDE_KM15),
        "using_ww": True,
    },
    verbose = False, debugging = False)

x_training_bkm10_xsec = xsecs.compute_cross_section(
    x_training_phi_points, lepton_helicity = 0.0, target_polarization = 0.0).real

x_validation_bkm10_xsec = xsecs.compute_cross_section(
    x_validation_phi_points, lepton_helicity = 0.0, target_polarization = 0.0).real

x_testing_bkm10_xsec = xsecs.compute_cross_section(
    x_testing_phi_points, lepton_helicity = 0.0, target_polarization = 0.0).real

x_training_bkm10_bsa = xsecs.compute_bsa(
    x_training_phi_points, target_polarization = 0.0).real

x_validation_bkm10_bsa = xsecs.compute_bsa(
    x_validation_phi_points, target_polarization = 0.0).real

x_testing_bkm10_bsa = xsecs.compute_bsa(
    x_testing_phi_points, target_polarization = 0.0).real

#### (3.4.2): Making the Cross Section Plot:

In [ ]:
data_vis_figure, data_vis_axis = plt.subplots(1, 1, figsize = (10, 10))

data_vis_axis.scatter(
    x_training_phi_points,
    x_training_bkm10_xsec,
    color = 'green',
    s = 4., 
    label = rf"(KM15) Training Points ($N = {len(x_training_phi_points)}$)")
data_vis_axis.scatter(
    x_validation_phi_points,
    x_validation_bkm10_xsec,
    color = 'orange',
    s = 4.,
    label = rf"(KM15) Validation Points ($N = {len(x_validation_phi_points)}$)")
data_vis_axis.scatter(
    x_testing_phi_points,
    x_testing_bkm10_xsec,
    color = 'red',
    s = 4.,
    label = rf"(KM15) Testing Points ($N = {len(x_testing_phi_points)}$)")

data_vis_axis.legend(fontsize = 20.)

data_vis_axis.set_xlabel(
    r"$\phi$ [radians]",
    fontsize = 20.)
data_vis_axis.set_ylabel(
    r"$d^{4}\sigma$ [nb / GeV$^{4}$]",
    fontsize = 20.)
data_vis_axis.set_title(
    rf"$d^{{4}}\sigma^{{UU}}$ vs. $\phi$, {this_kinematic_set_title_string}"
    "\n"
    f"(KM15): {km15_cff_string}",
    fontsize = 16
)

data_vis_axis.grid(
    visible = True,
    alpha = 0.35)

cross_section_visual_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/train_test_split_cross_section_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    data_vis_figure.savefig(
        f"{cross_section_visual_plotname}.{extension}",
        facecolor = 'white')
                            
plt.close(data_vis_figure)

#### (3.4.3): Making the BSA Plot:

In [ ]:
bsa_train_test_split_figure, bsa_train_test_split_axis = plt.subplots(1, 1, figsize = (10, 10))

bsa_train_test_split_axis.scatter(
    x_training_phi_points,
    x_training_bkm10_bsa,
    color = 'green',
    s = 4.,
    label = rf"(KM15) Training Points $N = {len(x_training_phi_points)}$")
bsa_train_test_split_axis.scatter(
    x_validation_phi_points,
    x_validation_bkm10_bsa,
    color = 'orange',
    s = 4.,
    label = rf"(KM15) Validation Points $N = {len(x_validation_phi_points)}$")
bsa_train_test_split_axis.scatter(
    x_testing_phi_points,
    x_testing_bkm10_bsa,
    color = 'red',
    s = 4.,
    label = rf"(KM15) Testing Points $N = {len(x_testing_phi_points)}$")

bsa_train_test_split_axis.legend(
    fontsize = 20.)

bsa_train_test_split_axis.set_xlabel(
    r"$\phi$ [radians]",
    fontsize = 20.)
bsa_train_test_split_axis.set_ylabel(
    r"BSA [unitless]",
    fontsize = 20.)
bsa_train_test_split_axis.set_title(
    rf"BSA vs. $\phi$, {this_kinematic_set_title_string}"
    "\n"
    f"(KM15): {km15_cff_string}",
    fontsize = 16)

bsa_visual_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/train_test_split_bsa_v{MAJOR_MINOR_NUMBER}"

bsa_train_test_split_axis.grid(
    visible = True,
    alpha = 0.35)

for extension in ['png', 'eps']:
    bsa_train_test_split_figure.savefig(
        f"{bsa_visual_plotname}.{extension}",
        facecolor = 'white')

plt.close(bsa_train_test_split_figure)

### (4.5): DNN Model Architecture:

In [ ]:
def cff_h_model():

    kinematics_inputs = tf.keras.Input(
        shape = (3,),
        name = "input_values")
    
    physics_input = tf.keras.Input(
        shape = (19,),
        name = "precomputed_physics")
    
    dnn_kinematic_inputs = tf.keras.layers.Lambda(
        lambda x: x[:, :3],
        name = "input_kinematics")(kinematics_inputs)
    
    hidden = tf.keras.layers.Dense(
        10,
        kernel_initializer = "he_normal",
        activation = "relu")(dnn_kinematic_inputs)
    
    hidden = tf.keras.layers.Dense(
        10,
        kernel_initializer = "he_normal",
        activation = "relu")(hidden)
    
    hidden = tf.keras.layers.Dense(
        10,
        kernel_initializer = "he_normal",
        activation = "relu")(hidden)
    
    hidden = tf.keras.layers.Dense(
        10,
        kernel_initializer = "he_normal",
        activation = "relu")(hidden)
    
    # Re[H], Im[H], Re[Ht], Im[Ht], Re[E], Im[E], Re[Et], Im[Et]
    cff_outputs = tf.keras.layers.Dense(
        8,
        activation = "linear",
        name = "cff_outputs")(hidden)
    
    full_model_outputs = tf.keras.layers.Concatenate(
        name = "physics_and_cffs")([
            cff_outputs, 
            physics_input])
    
    model = tf.keras.Model(
        inputs = [
            kinematics_inputs,
            physics_input
            ],
        outputs = full_model_outputs)

    model.compile(
        optimizer = tf.keras.optimizers.Adam(
            learning_rate = LEARNING_RATE
            ),
        loss = UnfoldedSimultaneousFitLoss(
            enabled_observables = enabled_observables,
            observable_weights = OBSERVABLE_WEIGHTS
        ),
        jit_compile = True,
        )
    
    # return model:
    return model

## (5): Fitting the Model:

### (5.1): Get a TF Summary of the DNN Model:

In [ ]:
cff_h_model().summary()

### (5.2): **Begin the Replica Method**

In [ ]:
for replica in range(NUMBER_OF_REPLICAS):
    
    replica_number = replica + 1

    start_dnn_compile_time = time.perf_counter()
    dnn_model = cff_h_model()
    print(f"[TIMING]: model construction time: {time.perf_counter() - start_dnn_compile_time:.2f} s")

    start_dnn_fitting_time = time.perf_counter()
    dnn_model_history = dnn_model.fit(
        x = {
            "input_values": x_training,
            "precomputed_physics": precomputed_physics_training
        },
        y = y_training,
        validation_data = (
            {
                "input_values": x_validation,
                "precomputed_physics": precomputed_physics_validation
            },
            y_validation
        ),
        epochs = NUMBER_OF_EPOCHS,
        batch_size = BATCH_SIZE,
        verbose = 0
    )
    print(f"[TIMING]: fitting time: {time.perf_counter() - start_dnn_fitting_time:.2f} s")

    number_of_epochs_run = len(dnn_model_history.epoch)
    print(f"[INFO]: The model ran for {number_of_epochs_run} epochs before early stopping.")

    start_dnn_save_time = time.perf_counter()
    dnn_model.save(
        replica_directory /
        f"replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.keras"
        )
    print(f"[TIMING]: DNN saving time: {time.perf_counter() - start_dnn_save_time:.2f} s")

    training_loss_data = dnn_model_history.history["loss"]
    validation_loss_data = dnn_model_history.history["val_loss"]

    start_dnn_evaluation_time = time.perf_counter()
    dnn_evaluation_statistics = dnn_model.evaluate(
        x = {
            "input_values": x_testing,
            "precomputed_physics": precomputed_physics_testing
            },
        y = y_testing,
        verbose = 0)
    print(f"[TIMING]: DNN evaluation time: {time.perf_counter() - start_dnn_evaluation_time:.2f} s")
    print(f"[INFO]: Test Loss for Replica {replica_number}: {dnn_evaluation_statistics}")

    # make DF with testing metrics:
    pd.DataFrame({
        'testing_loss': [dnn_evaluation_statistics], # https://stackoverflow.com/a/17840195 -> for why we need to cast it into a list!
    }).to_csv(
        replica_directory /
        f"replica_{replica_number}_loss_data.csv",
        index = False)

    # save npz with DNN training information:
    np.savez(
        file = replica_directory / f"replica_{replica_number}_losses_vs_epochs.npz",
        training_loss = training_loss_data,
        validation_loss = validation_loss_data
        )

    # make DF with DNN training information
    pd.DataFrame(dnn_model_history.history).to_csv(
        replica_directory / f"replica_{replica_number}_losses_vs_epochs.csv",
        index = False)

    predicted_outputs = dnn_model.predict(
        {
            "input_values": dnn_inputs,
            "precomputed_physics": physics_data,
        },
        verbose = 0
    )

    cff_predictions = predicted_outputs[:, :8]

    cff_h_real = cff_predictions[:, 0]
    cff_h_imag = cff_predictions[:, 1]
    cff_ht_real = cff_predictions[:, 2]
    cff_ht_imag = cff_predictions[:, 3]
    cff_e_real = cff_predictions[:, 4]
    cff_e_imag = cff_predictions[:, 5]
    cff_et_real = cff_predictions[:, 6]
    cff_et_imag = cff_predictions[:, 7]

    fe = compute_fe(t)
    fg = compute_fg(fe)
    f2 = compute_f2(t, fe, fg)
    f1 = compute_f1(fg, f2)
    epsilon = compute_epsilon(xb, q2)
    y_lep = compute_y(FIXED_K, q2, epsilon)
    xi = compute_skewness(xb, t, q2)
    tmin = compute_t_min(xb, q2, epsilon)
    tprime = compute_t_prime(t, tmin)
    k_tilde = compute_k_tilde(xb, q2, t, t_min, epsilon)
    kinematic_k = compute_k(q2, y, epsilon, k_tilde)
    kdd = compute_k_dot_delta(q2, xb, t, phi, epsilon, y, kinematic_k)
    p1 = prop_1(q2, kdd)
    p2 = prop_2(q2, t, kdd)
    cross_section_prediction = bkm10_cross_section(
        TEST_LEPTON_HELICITY, TEST_TARGET_POLARIZATION,
        q2, xb, t, epsilon, y_lep, xi, k, f1, f2, k_tilde, tprime, phi, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)

    bsa_prediction = bkm10_bsa(
        TEST_TARGET_POLARIZATION,
        q2, xb, t, epsilon, y_lep, xi, k, f1, f2, k_tilde, tprime, phi, p1, p2,
        cff_h_real, cff_ht_real, cff_e_real, cff_et_real, cff_h_imag, cff_ht_imag, cff_e_imag, cff_et_imag)

    np.savez(
        file = replica_directory / f"replica_{replica_number}_predictions.npz",
        t = t,
        xb = xb,
        q_squared = q2,
        phi = phi,

        cff_h_real = cff_predictions[:, 0],
        cff_h_imag = cff_predictions[:, 1],
        cff_ht_real = cff_predictions[:, 2],
        cff_ht_imag = cff_predictions[:, 3],
        cff_e_real = cff_predictions[:, 4],
        cff_e_imag = cff_predictions[:, 5],
        cff_et_real = cff_predictions[:, 6],
        cff_et_imag = cff_predictions[:, 7],

        cross_section = cross_section_prediction,
        bsa = bsa_prediction,
    )


    # [INFO]: this is called an "aggressive" memory cleanup:
    del dnn_model
    del dnn_model_history
    del training_loss_data
    del validation_loss_data
    del dnn_evaluation_statistics

    gc.collect()
    tf.keras.backend.clear_session()

### (5.3): Make Corresponding Loss Plots:

In [ ]:
# just do this for now hahahaha:
_REGULARIZER = 1e-21

for loss_file in sorted(
    glob.glob(
        f"./local/version_{MAJOR_MINOR_NUMBER}/replicas/replica_*_losses_vs_epochs.csv"
        )
    ):

    replica_match = re.search(r"replica_(\d+)_losses_vs_epochs\.csv", loss_file)

    if replica_match is None:
        print(f"[WARNING]: Could not determine replica number from {loss_file}")
        continue

    replica_number = int(replica_match.group(1))

    loss_information = pd.read_csv(loss_file)
    
    training_loss_data = loss_information["loss"].to_numpy()
    validation_loss_data = loss_information["val_loss"].to_numpy()

    number_of_epochs_run = len(loss_information)
    epoch_array = np.arange(number_of_epochs_run)

    testing_loss_information = pd.read_csv(
        f"./local/version_{MAJOR_MINOR_NUMBER}/replicas/replica_{replica_number}_loss_data.csv")

    testing_loss = testing_loss_information["testing_loss"].iloc[0]

    initial_loss = np.max(training_loss_data)

    curves_fig, curves_ax = plt.subplots(1, figsize = (8, 8))
    log_curves_fig, log_curves_ax = plt.subplots(1, figsize = (8, 8))

    curves_ax.plot(
        epoch_array,
        np.full(number_of_epochs_run, initial_loss),
        color = "red",
        label = "Initial Loss Value")
    
    curves_ax.plot(
        epoch_array,
        np.zeros(number_of_epochs_run),
        color = "green",
        label = r"Loss $= 0$")
    
    curves_ax.plot(
        epoch_array,
        training_loss_data,
        color = "blue",
        label = "Training Loss")
    
    curves_ax.plot(
        epoch_array,
        validation_loss_data,
        color = "purple",
        label = "Validation Loss")

    curves_ax.legend(fontsize = 15)

    curves_ax.set_xlabel(
        "Epoch",
        fontsize = 15)
    
    curves_ax.set_ylabel(
        "MSE",
        fontsize = 15)
    
    curves_ax.set_title(
        f"Replica {replica_number} Learning Curves\n(Test Loss $= {testing_loss:.3g}$)",
        fontsize = 15)
    
    log_curves_ax.plot(
        epoch_array,
        np.log(np.full(number_of_epochs_run, initial_loss)),
        color = "red",
        label = "Initial Loss Value")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(np.full(number_of_epochs_run, _REGULARIZER)),
        color = "green",
        label = r"Loss $= 0$")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(training_loss_data + _REGULARIZER),
        color = "blue",
        label = "Log Training Loss")
    
    log_curves_ax.plot(
        epoch_array,
        np.log(validation_loss_data + _REGULARIZER),
        color = "purple",
        label = "Log Validation Loss")

    log_curves_ax.legend(fontsize = 15)

    log_curves_ax.set_xlabel(
        "Epoch", 
        fontsize = 15)
    
    log_curves_ax.set_ylabel(
        "Log MSE Loss", 
        fontsize = 15)
    
    log_curves_ax.set_title(
        f"Replica {replica_number} Learning Curves\n(Test Loss $= {testing_loss:.3g}$)",
        fontsize = 15)

    for extension in ['png', 'eps']:
        curves_fig.savefig(
            learning_curves_directory /
            f"lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white')

        log_curves_fig.savefig(
            learning_curves_directory /
            f"log_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white')

    plt.close(curves_fig)
    plt.close(log_curves_fig)

    del curves_fig
    del log_curves_fig

## (6): DNN Predictions:

### (6.1): Prepare Smooth Input Data:

In [ ]:
range_of_t = np.linspace(x_training.min(), x_training.max())
range_of_x_b = np.linspace(x_training.min(), x_training.max())
range_of_q_squared = np.linspace(x_training.min(), x_training.max())
range_of_t

In [ ]:
replica_paths = sorted(
    replica_directory.glob(f"replica_*_v{MAJOR_MINOR_NUMBER}.keras")
)

replicas = [tf.keras.models.load_model(
    path,
    compile = False,
    safe_mode = False) for path in replica_paths]

print(f"[INFO]: Loaded {len(replicas)} replica models.")


assert len(replicas) == NUMBER_OF_REPLICAS, "[ASSERT]: Number of loaded replicas does not equal expected number"

In [ ]:
replicas_cross_predictions = []
replicas_bsa_predictions = []

replicas_h_real = []
replicas_h_imag = []
replicas_ht_real = []
replicas_ht_imag = []
replicas_e_real = []
replicas_e_imag = []
replicas_et_real = []
replicas_et_imag = []

for replica_index in range(1, NUMBER_OF_REPLICAS + 1):

    prediction_path = (
        replica_directory
        / f"replica_{replica_index}_predictions.npz"
    )

    prediction_information = np.load(prediction_path)
    replicas_h_real.append(prediction_information["cff_h_real"])
    replicas_h_imag.append(prediction_information["cff_h_imag"])
    replicas_ht_real.append(prediction_information["cff_ht_real"])
    replicas_ht_imag.append(prediction_information["cff_ht_imag"])
    replicas_e_real.append(prediction_information["cff_e_real"])
    replicas_e_imag.append(prediction_information["cff_e_imag"])
    replicas_et_real.append(prediction_information["cff_et_real"])
    replicas_et_imag.append(prediction_information["cff_et_imag"])
    replicas_cross_predictions.append(prediction_information["cross_section"])
    replicas_bsa_predictions.append(prediction_information["bsa"])

    prediction_information.close()

In [ ]:
def crunch_statistics(data):

    # huge dictionary of statistics
    statistics_dictionary = {
        'mean': np.mean(data, axis = 0),
        'std': np.std(data, axis = 0),
        'median': np.median(data, axis = 0),
        'min': np.min(data, axis = 0),
        'max': np.max(data, axis = 0)
    }

    for percentile in range(10, 50, 10):
        statistics_dictionary[f'p{percentile}'] = np.percentile(data, percentile, axis = 0)
        statistics_dictionary[f'p{100 - percentile}'] = np.percentile(data, 100 - percentile, axis = 0)

    return statistics_dictionary

# observable statistics:
xs_stats = crunch_statistics(replicas_cross_predictions)
bsa_stats = crunch_statistics(replicas_bsa_predictions)

replica_statistics_dataframe = pd.DataFrame({
    'k': FIXED_K,
    't': FIXED_T,
    'xb': FIXED_XB,
    'q_squared': FIXED_Q_SQUARED,
    'phi': phi_array_in_degrees,

    # TRUE CFF VALUES:
    "Re[H]": CFF_REAL_H_KM15, "Im[H]": CFF_IMAG_H_KM15,
    "Re[E]": CFF_REAL_E_KM15, "Im[E]": CFF_IMAG_E_KM15,
    "Re[Ht]": CFF_REAL_HT_KM15, "Im[Ht]": CFF_IMAG_HT_KM15,
    "Re[Et]": CFF_REAL_ET_KM15, "Im[Et]": CFF_IMAG_ET_KM15,

    # cross-section
    'mean_xs': xs_stats['mean'],
    'std_xs': xs_stats['std'],
    'min_xs': xs_stats['min'],
    'max_xs': xs_stats['max'],
    'p10_xs': xs_stats['p10'], 'p20_xs': xs_stats['p20'], 'p30_xs': xs_stats['p30'], 'p40_xs': xs_stats['p40'],
    'p60_xs': xs_stats['p60'], 'p70_xs': xs_stats['p70'], 'p80_xs': xs_stats['p80'], 'p90_xs': xs_stats['p90'],
    
    # BSA
    'mean_bsa': bsa_stats['mean'],
    'std_bsa': bsa_stats['std'],
    'min_bsa': bsa_stats['min'],
    'max_bsa': bsa_stats['max'],
    'p10_bsa': bsa_stats['p10'], 'p20_bsa': bsa_stats['p20'], 'p30_bsa': bsa_stats['p30'], 'p40_bsa': bsa_stats['p40'],
    'p60_bsa': bsa_stats['p60'], 'p70_bsa': bsa_stats['p70'], 'p80_bsa': bsa_stats['p80'], 'p90_bsa': bsa_stats['p90']
})

replica_statistics_dataframe.to_csv(
    data_directory / f"observable_preds_v{MAJOR_MINOR_NUMBER}.csv", 
    index = False)

cff_h_real_pred_per_replica = np.mean(
    replicas_h_real,
    axis = 1
)

cff_h_imag_pred_per_replica = np.mean(
    replicas_h_imag,
    axis = 1
)

cff_ht_real_pred_per_replica = np.mean(
    replicas_ht_real,
    axis = 1
)

cff_ht_imag_pred_per_replica = np.mean(
    replicas_ht_imag,
    axis = 1
)

cff_e_real_pred_per_replica = np.mean(
    replicas_e_real,
    axis = 1
)

cff_e_imag_pred_per_replica = np.mean(
    replicas_e_imag,
    axis = 1
)

cff_et_real_pred_per_replica = np.mean(
    replicas_et_real,
    axis = 1
)

cff_et_imag_pred_per_replica = np.mean(
    replicas_et_imag,
    axis = 1
)

replica_cffs_dataframe = pd.DataFrame({
    "ReH_pred": cff_h_real_pred_per_replica,
    "ImH_pred": cff_h_imag_pred_per_replica,

    "ReHt_pred": cff_ht_real_pred_per_replica,
    "ImHt_pred": cff_ht_imag_pred_per_replica,

    "ReE_pred": cff_e_real_pred_per_replica,
    "ImE_pred": cff_e_imag_pred_per_replica,

    "ReEt_pred": cff_et_real_pred_per_replica,
    "ImEt_pred": cff_et_imag_pred_per_replica,
})

replica_cffs_dataframe.to_csv(
    data_directory /
    f"cff_replica_average_preds_v{MAJOR_MINOR_NUMBER}.csv", 
    index = False)

Just checking out the array.

### (6.6): Figures of DNN $d^{4}\sigma$ Prediction

In [ ]:
fig2, ax2 = plt.subplots(1, figsize = (10, 7))

ax2.scatter(
    phi_array_in_radians, bkm10_unp_beam_unp_target_km15,
    s = 4., label = "BKM10 Prediction with KM15 CFFs", color = "blue")

ax2.plot(
    phi_array_in_radians,
    replica_statistics_dataframe['mean_xs'],
    label = r'Replica Average',
    color = "blue",
    linewidth = 0.5,
    linestyle = 'dashed')

ax2.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['max_xs'],
    y2 = replica_statistics_dataframe['min_xs'],
    label = r'Min/Max Bound',
    color = "lightgray",
    alpha = 0.2)

ax2.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p90_xs'],
    y2 = replica_statistics_dataframe['p10_xs'],
    label = r'10/90 \% Bound',
    color = "gray",
    alpha = 0.25)

ax2.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p80_xs'],
    y2 = replica_statistics_dataframe['p20_xs'],
    label = r'20/80 \% Bound',
    color = "gray",
    alpha = 0.3)

ax2.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p70_xs'],
    y2 = replica_statistics_dataframe['p30_xs'],
    label = r'30/70 \% Bound',
    color = "gray",
    alpha = 0.35)

ax2.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p60_xs'],
    y2 = replica_statistics_dataframe['p40_xs'],
    label = r'40/60 \% Bound',
    color = "gray",
    alpha = 0.4)

ax2.set_xlabel(r"$\phi$ [radians]", fontsize = 16)
ax2.set_ylabel(r"$d^{4}\sigma$ [nb / GeV$^{4}$]", fontsize = 16)
ax2.set_title(
    rf"$d^{{4}}\sigma^{{UU}}$ vs. $\phi$, {this_kinematic_set_title_string}"
    "\n"
    f"(KM15): {km15_cff_string}", fontsize = 16
)

ax2.legend()
plt.tight_layout()

replica_cross_section_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/dnn_xsec_vs_phi_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    fig2.savefig(
        f"{replica_cross_section_plotname}.{extension}",
        facecolor = 'white')
    
plt.close(fig2)

### (6.7): Figures of DNN $\text{BSA}$ Prediction

In [ ]:
fig3, ax3 = plt.subplots(1, figsize = (10, 7))

ax3.scatter(
    phi_array_in_radians, bkm10_bsa_km15,
    s = 4., label = "BKM10 Prediction with KM15 CFFs", color = "blue")

ax3.plot(
    phi_array_in_radians,
    replica_statistics_dataframe['mean_bsa'],
    label = r'Replica Average',
    color = "blue",
    linewidth = 0.5,
    linestyle = 'dashed')

ax3.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['max_bsa'],
    y2 = replica_statistics_dataframe['min_bsa'],
    label = r'Min/Max Bound',
    color = "lightgray",
    alpha = 0.2)

ax3.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p90_bsa'],
    y2 = replica_statistics_dataframe['p10_bsa'],
    label = r'10/90 \% Bound',
    color = "gray",
    alpha = 0.25)

ax3.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p80_bsa'],
    y2 = replica_statistics_dataframe['p20_bsa'],
    label = r'20/80 \% Bound',
    color = "gray",
    alpha = 0.3)

ax3.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p70_bsa'],
    y2 = replica_statistics_dataframe['p30_bsa'],
    label = r'30/70 \% Bound',
    color = "gray",
    alpha = 0.35)

ax3.fill_between(
    x = phi_array_in_radians,
    y1 = replica_statistics_dataframe['p60_bsa'],
    y2 = replica_statistics_dataframe['p40_bsa'],
    label = r'40/60 \% Bound',
    color = "gray",
    alpha = 0.4)

ax3.set_xlabel(r"$\phi$ [radians]", fontsize = 16)
ax3.set_ylabel(r"BSA [unitless]", fontsize = 16)
ax3.set_title(
    rf"BSA vs. $\phi$, {this_kinematic_set_title_string}"
    "\n"
    f"(KM15): {km15_cff_string}", fontsize = 16)

ax3.legend()
fig3.tight_layout()

replica_bsa_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/dnn_bsa_vs_phi_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    fig3.savefig(
        f"{replica_bsa_plotname}.{extension}",
        facecolor = 'white')

plt.close(fig3)

### (6.8): Compute Mean $Re[\mathcal{H}]$ and $Im[\mathcal{H}]$ using Replicas:

In [ ]:
def make_cff_plot_label(
        cff_label: str
    ):
    """
    I am going to customize LaTeX representation of a given CFF
    based on the string version that I received from a datafile.
    """

    # why does this work? because the variable is a STRING
    real_or_imag = cff_label[:2]
    cff_name = cff_label[2:]

    component = { "Re": "Re", "Im": "Im" }[real_or_imag]

    cff_symbol = {
        "H": r"\mathcal{H}",
        "Ht": r"\widetilde{\mathcal{H}}",
        "E": r"\mathcal{E}",
        "Et": r"\widetilde{\mathcal{E}}",
    }[cff_name]

    return rf"{component}$[{cff_symbol}]$"

_NUMBER_OF_STDDEVS = 4.
_NUMBER_OF_HISTOGRAM_BINS = 30
_NUMBER_OF_GAUSSIAN_POINTS = 200

km15_values = {
    "ReH": CFF_REAL_H_KM15,
    "ImH": CFF_IMAG_H_KM15,
    "ReE": CFF_REAL_E_KM15,
    "ImE": CFF_IMAG_E_KM15,
    "ReHt": CFF_REAL_HT_KM15,
    "ImHt": CFF_IMAG_HT_KM15,
    "ReEt": CFF_REAL_ET_KM15,
    "ImEt": CFF_IMAG_ET_KM15,
}

cff_h_km15 = complex(CFF_REAL_H_KM15, CFF_IMAG_H_KM15)
cff_e_km15 = complex(CFF_REAL_E_KM15, CFF_IMAG_E_KM15)
cff_ht_km15 = complex(CFF_REAL_HT_KM15, CFF_IMAG_HT_KM15)
cff_et_km15 = complex(CFF_REAL_ET_KM15, CFF_IMAG_ET_KM15)

km15_cff_string = (
    rf"$\mathcal{{H}} = {cff_h_km15:.3f}$, "
    rf"$\mathcal{{E}} = {cff_e_km15:.3f}$, "
    rf"$\widetilde{{\mathcal{{H}}}} = {cff_ht_km15:.3f}$, "
    rf"$\widetilde{{\mathcal{{E}}}} = {cff_et_km15:.3f}$"
)

for cff_label in km15_values:
        
    corresponding_key = f"{cff_label}_pred"

    if corresponding_key not in replica_cffs_dataframe.columns:
        continue

    cff_prediction_per_replica = replica_cffs_dataframe[corresponding_key]
    
    cff_mean, cff_stddev = norm.fit(cff_prediction_per_replica)
    
    cff_km15_value = km15_values[cff_label]
    
    gaussian_x_values = np.linspace(
        cff_mean - _NUMBER_OF_STDDEVS * cff_stddev,
        cff_mean + _NUMBER_OF_STDDEVS * cff_stddev,
        _NUMBER_OF_GAUSSIAN_POINTS
    )
    
    kinematic_title = (
        rf"$k = {FIXED_K:.3f}$ GeV, "
        rf"$x_B = {FIXED_XB:.3f}$, "
        rf"$t = {FIXED_T:.3f}$ GeV$^2$, "
        rf"$Q^2 = {FIXED_Q_SQUARED:.3f}$ GeV$^2$"
        )
    
    cff_figure, cff_axis = plt.subplots(1, 1, figsize = (10, 8))
    
    cff_axis.hist(
        cff_prediction_per_replica,
        bins = _NUMBER_OF_HISTOGRAM_BINS,
        alpha = 0.6,
        color = "skyblue",
        edgecolor = "black")
    
    cff_axis.plot(
        gaussian_x_values,
        norm.pdf(
            gaussian_x_values,
            cff_mean,
            cff_stddev
        ),
        color = "red",
        linestyle = "--",
        label = (
            fr"Gaussian Fit: $\mu = {cff_mean:.3f}$, $\sigma = {cff_stddev:.3f}$"
        )
    )
    
    cff_axis.axvline(
        cff_km15_value,
        color = "green",
        linestyle = "-",
        linewidth = 2.0,
        label = f"KM15: {cff_km15_value:.3f}")
    
    cff_axis.set_ylabel(
        "Frequency",
        rotation = 90.,
        fontsize = 16.0)
    
    cff_axis.set_xlabel(
        make_cff_plot_label(cff_label),
        fontsize = 16.0)
    
    cff_axis.set_title(
        rf"{make_cff_plot_label(cff_label)} Distribution, "
        rf"{kinematic_title}"
        "\n"
        rf"(KM15): {km15_cff_string}",
        fontsize = 16.0)
    
    cff_axis.legend(fontsize = 16.0)
    
    cff_axis.text(
        0.00, -0.05,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}",
        transform = cff_axis.transAxes)
    
    cff_figure.tight_layout()
    
    for extension in ["png", "eps"]:
        cff_figure.savefig(
            plots_directory /
            f"cff_{cff_label}_fits_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = "white",
            transparent = False
        )
    
    plt.close(cff_figure)

## (7): Use Predicted CFF Values to Compare Observables:

### (7.1): Use Predicted $\mathcal{H}$ Values in BKM10 **Cross-Section and BSA** Prediction:

In [ ]:
predicted_h_bkm10 = DifferentialCrossSection(
    configuration = {
        "kinematics": BKM10Inputs(
            lab_kinematics_k = FIXED_K,
            squared_Q_momentum_transfer  = FIXED_Q_SQUARED,
            x_Bjorken = FIXED_XB,
            squared_hadronic_momentum_transfer_t = FIXED_T),
        "cff_inputs": CFFInputs(
            compton_form_factor_h = complex(
                cff_h_real_pred_per_replica, 
                cff_h_imag_pred_per_replica),
            compton_form_factor_h_tilde = complex(
                cff_ht_real_pred_per_replica, 
                cff_ht_imag_pred_per_replica),
            compton_form_factor_e = complex(
                cff_e_real_pred_per_replica, 
                cff_e_imag_pred_per_replica),
            compton_form_factor_e_tilde = complex(
                cff_et_real_pred_per_replica, 
                cff_et_imag_pred_per_replica),),
        "using_ww": True
    },
    verbose = False,
    debugging = False)

bkm10_cross_sections_dnn = predicted_h_bkm10.compute_cross_section(
    phi_array_in_radians,
    lepton_helicity = 0.0,
    target_polarization = 0.0).real

bkm10_bsa_dnn = predicted_h_bkm10.compute_bsa(
    phi_array_in_radians,
    target_polarization = 0.0).real

### (7.2): Overlay the $d^{4}\sigma$ Computed from DNN Fit of $\mathcal{H}$

In [ ]:
post_cff_fit_xsec_figure, post_cff_fit_xsec_axis = plt.subplots(nrows = 2, ncols = 1, 
    gridspec_kw = {'height_ratios': [3, 1]},
    figsize = (7, 7))

post_cff_fit_xsec_axis[1].plot( - bkm10_unp_beam_unp_target_km15, color = 'gray', label = "Residuals")

post_cff_fit_xsec_axis[0].scatter(
    phi_array_in_radians, bkm10_unp_beam_unp_target_km15,
    s = 4., label = "BKM10 Prediction with KM15 CFFs", color = "blue")

post_cff_fit_xsec_axis[0].plot(
    phi_array_in_radians,
    mean_xs,
    label = r'Replica Average',
    color = "blue",
    linewidth = 0.5,
    linestyle = 'dashed')

post_cff_fit_xsec_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = xs_max,
    y2 = xs_min,
    label = r'Min/Max Bound',
    color = "lightgray",
    alpha = 0.2)

post_cff_fit_xsec_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = xs_percentile_90,
    y2 = xs_percentile_10,
    label = r'10/90 \% Bound',
    color = "gray",
    alpha = 0.25)

post_cff_fit_xsec_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = xs_percentile_80,
    y2 = xs_percentile_20,
    label = r'20/80 \% Bound',
    color = "gray",
    alpha = 0.3)

post_cff_fit_xsec_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = xs_percentile_70,
    y2 = xs_percentile_30,
    label = r'30/70 \% Bound',
    color = "gray",
    alpha = 0.35)

post_cff_fit_xsec_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = xs_percentile_60,
    y2 = xs_percentile_40,
    label = r'40/60 \% Bound',
    color = "gray",
    alpha = 0.4)

post_cff_fit_xsec_axis[0].set_xlabel(r"$\phi$ [radians]", fontsize = 16)
post_cff_fit_xsec_axis[0].set_ylabel(r"$d^{4}\sigma$ [nb / GeV$^{4}$]", fontsize = 16)
post_cff_fit_xsec_axis[1].set_ylabel(r"Residuals", fontsize = 16)
post_cff_fit_xsec_axis[0].set_title(f"{this_kinematic_set_title_string}\n(KM15): {km15_cff_string}")

post_cff_fit_xsec_axis[0].legend()

post_cff_fit_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/cross_section_comparison_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    post_cff_fit_xsec_figure.savefig(
        f"{post_cff_fit_plotname}.{extension}",
        facecolor = 'white')

plt.close(post_cff_fit_xsec_figure)

### (7.3): Overlay the $\text{BSA}$ Computed from DNN Fit of $\mathcal{H}$

In [ ]:
post_cff_fit_bsa_figure, post_cff_fit_bsa_axis = plt.subplots(
    nrows = 2, ncols = 1,  gridspec_kw = {'height_ratios': [3, 1]}, figsize = (7, 7))

post_cff_fit_bsa_axis[1].plot(mean_bsa - bkm10_bsa_km15, color = 'gray', label = "Residuals")

post_cff_fit_bsa_axis[0].scatter(
    phi_array_in_radians, bkm10_bsa_km15,
    s = 4., label = "BKM10 Prediction with KM15 CFFs", color = "blue")

post_cff_fit_bsa_axis[0].plot(
    phi_array_in_radians,
    mean_bsa,
    label = r'Replica Average',
    color = "blue",
    linewidth = 0.5,
    linestyle = 'dashed')

post_cff_fit_bsa_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = bsa_max,
    y2 = bsa_min,
    label = r'Min/Max Bound',
    color = "lightgray",
    alpha = 0.2)

post_cff_fit_bsa_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = bsa_percentile_90,
    y2 = bsa_percentile_10,
    label = r'10/90 \% Bound',
    color = "gray",
    alpha = 0.25)

post_cff_fit_bsa_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = bsa_percentile_80,
    y2 = bsa_percentile_20,
    label = r'20/80 \% Bound',
    color = "gray",
    alpha = 0.3)

post_cff_fit_bsa_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = bsa_percentile_70,
    y2 = bsa_percentile_30,
    label = r'30/70 \% Bound',
    color = "gray",
    alpha = 0.35)

post_cff_fit_bsa_axis[0].fill_between(
    x = phi_array_in_radians,
    y1 = bsa_percentile_60,
    y2 = bsa_percentile_40,
    label = r'40/60 \% Bound',
    color = "gray",
    alpha = 0.4)

post_cff_fit_bsa_axis[0].set_xlabel(r"$\phi$ [radians]", fontsize = 16)
post_cff_fit_bsa_axis[0].set_ylabel(r"BSA", fontsize = 16)
post_cff_fit_bsa_axis[1].set_ylabel(r"Residuals", fontsize = 16)
post_cff_fit_bsa_axis[0].set_title(f"{this_kinematic_set_title_string}\n(KM15): {km15_cff_string}")

post_cff_fit_bsa_axis[0].legend()

post_cff_fit_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/bsa_comparison_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    post_cff_fit_bsa_figure.savefig(
        f"{post_cff_fit_plotname}.{extension}",
        facecolor = 'white')

plt.close(post_cff_fit_bsa_figure)

### (7.4): Construct a Rough Correlation Plot

In [ ]:
cff_h_correlation_figure, cff_h_correlation_axis = plt.subplots(1, figsize = (7, 7))

cff_h_correlation_axis.scatter(
    cff_h_real_pred_per_replica, cff_h_imag_pred_per_replica,
    s = 4., label = "BKM10 Prediction with KM15 CFFs", color = "blue")

cff_h_correlation_axis.set_xlabel(r"Re$[\mathcal{H}]$", fontsize = 16)
cff_h_correlation_axis.set_ylabel(r"Im$[\mathcal{H}]$", fontsize = 16)
cff_h_correlation_axis.set_title(f"{this_kinematic_set_title_string}\n(KM15): {km15_cff_string}")

cff_h_correlation_axis.legend()

cff_correlation_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/cff_h_correlation_v{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    cff_h_correlation_figure.savefig(
        f"{cff_correlation_plotname}.{extension}",
        facecolor = 'white')
    
plt.close(cff_h_correlation_figure)

In [ ]:
print(f"[INFO]: Shape of Re[H]: {np.shape(cff_h_real_pred_per_replica)}")
print(f"[INFO]: Shape of Im[H]: {np.shape(cff_h_imag_pred_per_replica)}")
print(f"[INFO]: Shape of Re[Ht]: {np.shape(cff_ht_real_pred_per_replica)}")
print(f"[INFO]: Shape of Im[Ht]: {np.shape(cff_ht_imag_pred_per_replica)}")

cff_corner_plot_data = np.column_stack([
    cff_h_real_pred_per_replica,
    cff_h_imag_pred_per_replica,
    cff_ht_real_pred_per_replica,
    cff_ht_imag_pred_per_replica
])

print(f"[INFO]: Shape of the corner plot data {cff_corner_plot_data.shape}")

cff_corner_plot = corner.corner(
    cff_corner_plot_data,
    labels = [
        r"Re$[\mathcal{H}]$",
        r"Im$[\mathcal{H}]$",
        r"Re$[\tilde{\mathcal{H}}]$",
        r"Im$[\tilde{\mathcal{H}}]$",
        ],
    plot_datapoints = True,
    show_titles = True,
    title_kwargs = {"fontsize": 14},
    label_kwargs = {"fontsize": 14},
    hist_kwargs = {
        'fc': 'skyblue',
        'ec': 'black',
        'histtype': 'bar',
        'lw': '0.9',
        },
    data_kwargs = {
        "alpha": 0.5, 
        "label": "BKM10 Prediction with KM15 CFFs"
        },
    title_fmt = ".3g",
    figsize = (10, 10)
)

cff_corner_plotname = f"./local/version_{MAJOR_MINOR_NUMBER}/plots/cff_h_corner_plot_{MAJOR_MINOR_NUMBER}"

for extension in ['png', 'eps']:
    cff_corner_plot.savefig(
        f"{cff_correlation_plotname}.{extension}",
        facecolor = 'white')